# 02.2 — VP-SGLD 전 파라미터 sweep + M-vs-N (세션 단위 관리)

**하는 일**:

1. `SWEEPS` 에 정의된 **모든 축** (β, η, τ, σ_start, n_steps, ignore_variance) 를 OAT sweep
2. 각 (axis, value) 마다 **N=N_HIGH 로 SGLD 1 번** 돌리고 trajectory 저장
3. trajectory subset 평가로 **M-vs-N 곡선** 만들기 (~수 초)
4. 모든 결과는 **세션 폴더 (timestamp + tag)** 안에 격리 → sweep 폴더가 무한히 쌓이지 않음

**저장 구조**:
```
{ENSEMBLE_ROOT}/comparisons/sessions/{session_ts}_{session_tag}/
├── session_config.json          # 이 세션의 SWEEPS/FIXED 스냅샷
├── summary.csv                   # 전체 sweep 의 final M 등 집계
├── M_vs_N_overlay.png            # axis 별 곡선 겹쳐 그린 다중 패널
└── sweeps/
    ├── beta__0.01/
    │   ├── run_config.json       # 이 sweep 의 full cfg
    │   ├── diag_raw_c{c}.npz     # per-step diag + trajectory
    │   └── M_vs_N_c{c}.png       # 이 config 의 M-vs-N
    ├── beta__1.0/
    ├── eta__0.05/
    └── ...
```

**nb 03 연동**: 생성된 sweep 폴더 이름 (`beta__100.0` 등) 으로 원하는 config 만 선택해 classifier 비교 가능.

## 0. 설정 — ARGS_STR + SWEEPS dict

In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os, sys, json, argparse, shlex, time, datetime as _dt
os.chdir('/home/work/JooKyung/TabEBM')
sys.path.insert(0, 'experiments'); sys.path.insert(0, 'src')
from pathlib import Path

import numpy as np, pandas as pd, torch
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as _mp
try: _mp.set_start_method('spawn', force=True)
except RuntimeError: pass
import matplotlib.pyplot as plt

from tabebm.vp_sgld import (
    vp_sgld_from_ensemble,
    ensemble_score_var_fn, compute_beta_scale,
)

# ======================================================================
#  ARGS — CLI-style
# ======================================================================
ARGS_STR = '''
    # --- target ensemble ---
    --ensemble-root experiments/ebms/20260417_201751_Distance_EBM

    # --- FIXED (sweep 되지 않는 축의 기본값) ---
    --classes 0 1
    --n-high 500
    --n-steps 50
    --beta 1000000 --eta 0.05 --tau 1.0 --sigma-start 0.1
    --auto-beta
    --seed 0 --gpus 0 1 2 3     # sweep 을 이 GPU 들에 병렬 분산

    # --- 세션 태그 (식별용, 짧게) ---
    --session-tag all_axes_baseline
'''

parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--ensemble-root', type=Path, required=True)
parser.add_argument('--classes', type=int, nargs='+', default=[0, 1])
parser.add_argument('--n-high', type=int, default=500)
parser.add_argument('--n-steps', type=int, default=50)
parser.add_argument('--beta', type=float, default=1.0)
parser.add_argument('--eta', type=float, default=0.05)
parser.add_argument('--tau', type=float, default=1.0)
parser.add_argument('--sigma-start', type=float, default=0.1)
parser.add_argument('--auto-beta', action='store_true', default=False)
parser.add_argument('--no-auto-beta', dest='auto_beta', action='store_false')
parser.add_argument('--seed', type=int, default=0)
parser.add_argument('--gpus', type=int, nargs='+', default=[0])
parser.add_argument('--session-tag', type=str, default='sweep')

argv = []
for line in ARGS_STR.splitlines():
    line = line.split('#', 1)[0].strip()
    if line: argv.extend(shlex.split(line))
args = parser.parse_args(argv)

assert (args.ensemble_root / 'c0' / 'meta.json').exists(), \
    f'no c0/meta.json in {args.ensemble_root}'

# ======================================================================
#  SWEEPS — 축별로 돌릴 값들. baseline (FIXED 그대로) 은 항상 자동 포함.
#  원하지 않는 축은 주석 처리.
# ======================================================================
SWEEPS = {
    'beta':            [10000000, 100000000, 1000000000, 10000000000, 100000000000],        # β preconditioner 강도
    # 'eta':             [0.01, 0.1, 0.2, 1.0],              # step size
    # 'tau':             [0.1, 0.5, 2.0, 10.0],                    # noise temperature
    # 'sigma_start':     [0.01, 0.5, 2.0, 5.0],                # init perturbation
    # 'n_steps':         [100],                        # SGLD step 수
    'ignore_variance': [False, True],                        # M=I 강제 toggle
    # 'auto_beta':     [False, True],                         # (필요하면 해제)
}

# ======================================================================
#  M-vs-N scan 파라미터 — trajectory subset 에서 M 계산
# ======================================================================
N_SWEEP = sorted(set(
    list(range(2, 21)) +                                  # 2..20 전부
    [int(round(n)) for n in np.logspace(np.log10(21), np.log10(args.n_high), 30)]
))
EVAL_AT_STEPS = [0]   # M 평가 step (대부분 step=0 하나로 충분, final 도 원하면 추가)

# ======================================================================
#  세션 디렉토리 생성 — 이 세션의 모든 sweep 은 여기 격리
# ======================================================================
SESSION_TS = _dt.datetime.now().strftime('%Y%m%d_%H%M%S')
SESSION_DIR = args.ensemble_root / 'comparisons' / 'sessions' / f'{SESSION_TS}_{args.session_tag}'
SWEEPS_DIR = SESSION_DIR / 'sweeps'
SWEEPS_DIR.mkdir(parents=True, exist_ok=True)

(SESSION_DIR / 'session_config.json').write_text(json.dumps({
    'ensemble_root': str(args.ensemble_root),
    'ARGS_parsed': vars(args),
    'SWEEPS': SWEEPS,
    'N_SWEEP': N_SWEEP,
    'EVAL_AT_STEPS': EVAL_AT_STEPS,
    'session_ts': SESSION_TS,
}, indent=2, default=str))

# sweep 수 계산
total_configs = 1 + sum(len(v) for v in SWEEPS.values())    # 1 baseline + per-axis values

print(f'ENSEMBLE_ROOT : {args.ensemble_root}')
print(f'SESSION_DIR   : {SESSION_DIR}')
print(f'GPUS          : {args.gpus}  (PARALLEL={len(args.gpus) > 1})')
print(f'FIXED         : β={args.beta}, η={args.eta}, τ={args.tau}, '
       f'σ_start={args.sigma_start}, T={args.n_steps}, auto_β={args.auto_beta}')
print(f'SWEEPS axes   : {list(SWEEPS.keys())}  ({total_configs} configs: 1 baseline + sweep values)')
print(f'N_SWEEP       : {len(N_SWEEP)} points  ({N_SWEEP[:5]} ... {N_SWEEP[-3:]})')
print(f'예상 시간 (병렬): {total_configs} × |classes|={len(args.classes)} tasks / {len(args.gpus)} GPUs × ~44s ≈ {total_configs*len(args.classes)*44 / max(len(args.gpus),1) /60:.1f} 분')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
ENSEMBLE_ROOT : experiments/ebms/20260417_201751_Distance_EBM
SESSION_DIR   : experiments/ebms/20260417_201751_Distance_EBM/comparisons/sessions/20260417_202917_all_axes_baseline
GPUS          : [0, 1, 2, 3]  (PARALLEL=True)
FIXED         : β=1000000.0, η=0.05, τ=1.0, σ_start=0.1, T=50, auto_β=True
SWEEPS axes   : ['beta', 'ignore_variance']  (8 configs: 1 baseline + sweep values)
N_SWEEP       : 49 points  ([2, 3, 4, 5, 6] ... [402, 448, 500])
예상 시간 (병렬): 8 × |classes|=2 tasks / 4 GPUs × ~44s ≈ 2.9 분


## 1. SGLD 돌리기 — baseline + 각 sweep config

각 config 마다 `sweeps/{axis}__{value}/` 폴더 생성 후 trajectory + diag 저장.

In [3]:
# ProcessPool 으로 전환 — GIL 우회 진짜 4-GPU 병렬 (8x faster)
# worker 함수는 standalone script 에서 import (spawn 시 picklable 필요)
from experiments.run_sweep_processpool import _sgld_worker

def resolve_cfg(overrides: dict) -> dict:
    '''FIXED (args) 위에 overrides 덮어씌운 dict.'''
    return {
        'n_high': args.n_high,
        'beta': overrides.get('beta', args.beta),
        'eta': overrides.get('eta', args.eta),
        'tau': overrides.get('tau', args.tau),
        'sigma_start': overrides.get('sigma_start', args.sigma_start),
        'n_steps': overrides.get('n_steps', args.n_steps),
        'auto_beta': overrides.get('auto_beta', args.auto_beta),
        'ignore_variance': overrides.get('ignore_variance', False),
        'seed': args.seed,
    }

def cfg_folder_name(overrides: dict) -> str:
    if not overrides:
        return 'baseline'
    (axis, val), = overrides.items()
    return f'{axis}__{val}'

# 모든 (overrides, folder) + baseline (skip dedup 옵션)
SKIP_BASELINE_DUP = True   # sweep value 가 FIXED baseline 과 동일하면 skip
all_configs = [({}, 'baseline')]
for axis, values in SWEEPS.items():
    baseline_val = getattr(args, axis.replace('-', '_'), None)
    for v in values:
        if SKIP_BASELINE_DUP and v == baseline_val:
            print(f'[skip dedup] {axis}={v} == baseline → 건너뜀')
            continue
        all_configs.append(({axis: v}, cfg_folder_name({axis: v})))

# folder + run_config.json 미리 생성
for overrides, folder in all_configs:
    cfg = resolve_cfg(overrides)
    out_dir = SWEEPS_DIR / folder
    out_dir.mkdir(exist_ok=True)
    (out_dir / 'run_config.json').write_text(json.dumps({
        'overrides': overrides, 'cfg': cfg, 'folder': folder,
    }, indent=2, default=str))

# task 펼치기 — ProcessPool worker 가 받는 tuple 형식
tasks = []
for ci, (overrides, folder) in enumerate(all_configs):
    cfg = resolve_cfg(overrides)
    for c in args.classes:
        gpu = args.gpus[(ci * len(args.classes) + c) % len(args.gpus)]
        tasks.append((len(tasks), str(args.ensemble_root), folder, c, gpu, cfg, str(SWEEPS_DIR)))

PARALLEL = len(args.gpus) > 1
print(f'\nRunning {len(tasks)} tasks ({len(all_configs)} configs × {len(args.classes)} classes)  '
       f'on {len(args.gpus)} GPUs  (engine=ProcessPool, PARALLEL={PARALLEL})\n')

import sys
def _fmt_time(s):
    s = int(s); m, s = divmod(s, 60); h, m = divmod(m, 60)
    return f'{h:d}h{m:02d}m{s:02d}s' if h else f'{m:d}m{s:02d}s' if m else f'{s:d}s'

def _print_progress(done, total, r, t_total):
    elapsed = time.time() - t_total
    rate = elapsed / max(done, 1)
    eta = rate * (total - done) / max(len(args.gpus), 1)   # 병렬 보정
    pct = done / total * 100
    bar_w = 24
    filled = int(bar_w * done / total)
    bar = '█' * filled + '░' * (bar_w - filled)
    print(f'  [{bar}] {pct:5.1f}%  {done:>2d}/{total}  '
           f'{r["folder"]:<28} c{r["class_c"]} gpu={r["gpu"]} pid={r["pid"]}  '
           f'({r["dt"]:5.1f}s)  M={r["M_init"]:.3f}→{r["M_final"]:.3f}  '
           f'| elapsed {_fmt_time(elapsed)}  ETA {_fmt_time(eta)}',
           flush=True)

t_total = time.time()
results_log = []
total_tasks = len(tasks)
print(f'▶ START — {total_tasks} tasks on {len(args.gpus)} GPUs (ProcessPool)\n', flush=True)

if PARALLEL:
    with ProcessPoolExecutor(max_workers=len(args.gpus)) as ex:
        futures = {ex.submit(_sgld_worker, t): t for t in tasks}
        for i, fut in enumerate(as_completed(futures), start=1):
            r = fut.result()
            results_log.append(r)
            _print_progress(i, total_tasks, r, t_total)
else:
    for i, t in enumerate(tasks, start=1):
        r = _sgld_worker(t)
        results_log.append(r)
        _print_progress(i, total_tasks, r, t_total)

dt_total = time.time() - t_total
sum_dt = sum(r['dt'] for r in results_log)
print(f'\n✓ DONE — wallclock {_fmt_time(dt_total)}   '
      f'serial-equivalent {_fmt_time(sum_dt)}   '
      f'speedup ~{sum_dt/max(dt_total,0.01):.1f}x', flush=True)



Running 16 tasks (8 configs × 2 classes)  on 4 GPUs  (engine=ProcessPool, PARALLEL=True)

▶ START — 16 tasks on 4 GPUs (ProcessPool)






  [█░░░░░░░░░░░░░░░░░░░░░░░]   6.2%   1/16  beta__10000000               c1 gpu=3 pid=817600  ( 89.3s)  M=0.000→0.000  | elapsed 1m32s  ETA 5m45s

  [███░░░░░░░░░░░░░░░░░░░░░]  12.5%   2/16  beta__10000000               c0 gpu=2 pid=817599  ( 90.7s)  M=0.000→0.000  | elapsed 1m33s  ETA 2m43s

  [████░░░░░░░░░░░░░░░░░░░░]  18.8%   3/16  baseline                     c1 gpu=1 pid=817598  ( 91.6s)  M=0.000→0.000  | elapsed 1m34s  ETA 1m42s

  [██████░░░░░░░░░░░░░░░░░░]  25.0%   4/16  baseline                     c0 gpu=0 pid=817595  ( 93.8s)  M=0.000→0.000  | elapsed 1m36s  ETA 1m12s





  [███████░░░░░░░░░░░░░░░░░]  31.2%   5/16  beta__1000000000             c0 gpu=2 pid=817598  ( 72.9s)  M=0.000→0.000  | elapsed 2m47s  ETA 1m31s

  [█████████░░░░░░░░░░░░░░░]  37.5%   6/16  beta__100000000              c1 gpu=1 pid=817599  ( 78.1s)  M=0.000→0.000  | 

## 1-bis. 개별 config 실행 — sweep 없이 GPU 하나씩 원하는 거만

**용도**: sweep 전체 돌리지 않고, 관심 있는 설정 1~4 개만 병렬로 빠르게 비교.

- 각 GPU 에 하나의 config 배정 (`CUSTOM_CONFIGS[gpu_id] = {...}`)
- 필요없는 GPU 는 **주석 처리** (그 GPU 는 이 셀에서 놀게 됨)
- **전체 주석** 하면 이 셀은 아무것도 안 함 (건너뛰기)
- 결과는 sweep 과 **동일한 `sweeps/<tag>/` 폴더 구조** 로 저장 → § 2, § 3, § 4 재사용
- 기존 sweep 결과와 **공존** (overwrite 아님, 태그명 겹치지만 않으면)

In [ ]:
# ======================================================================
#  CUSTOM_CONFIGS — GPU 당 하나씩 (ProcessPool 진짜 병렬)
#  - key: GPU 번호 (args.gpus 에 있는 것만 유효)
#  - value: dict(beta, eta, tau, sigma_start, n_steps, auto_beta, ignore_variance, tag)
#  - 'tag' 는 sweeps/<tag>/ 폴더 이름. sweep 과 겹치지 않게 고유로.
#  - 필요없는 GPU 는 줄 자체를 주석 처리. 전체 비우면 이 셀 skip.
# ======================================================================
CUSTOM_CONFIGS = {
    # 0: dict(beta=100.0, eta=0.05, tau=1.0, sigma_start=0.1, n_steps=50,
    #          auto_beta=True, ignore_variance=False, tag='custom_b100_g0'),

    # 1: dict(beta=1.0, eta=0.05, tau=1.0, sigma_start=0.5, n_steps=50,
    #         auto_beta=True, ignore_variance=False, tag='custom_s05_g1'),

    # 2: dict(beta=1.0, eta=0.2, tau=1.0, sigma_start=0.1, n_steps=50,
    #         auto_beta=True, ignore_variance=False, tag='custom_eta02_g2'),

    # 3: dict(beta=1.0, eta=0.05, tau=0.1, sigma_start=0.1, n_steps=50,
    #         auto_beta=True, ignore_variance=True, tag='custom_ig_tau01_g3'),
}

# GPU 번호 validity 체크
bad_gpus = set(CUSTOM_CONFIGS) - set(args.gpus)
assert not bad_gpus, f'CUSTOM_CONFIGS 에 args.gpus 에 없는 GPU 지정됨: {bad_gpus}  (args.gpus={args.gpus})'

if not CUSTOM_CONFIGS:
    print('[skip] CUSTOM_CONFIGS 비어있음 — 개별 실행 건너뜀')
else:
    # tag 중복 체크
    tags = [c['tag'] for c in CUSTOM_CONFIGS.values()]
    assert len(set(tags)) == len(tags), f'tag 중복: {tags}'
    existing = [t for t in tags if (SWEEPS_DIR / t).exists()]
    if existing:
        print(f'⚠  이미 존재하는 tag (덮어씀): {existing}')

    print(f'Custom parallel — {len(CUSTOM_CONFIGS)} configs on GPUs {sorted(CUSTOM_CONFIGS)}\n')
    for gpu, cfg in CUSTOM_CONFIGS.items():
        print(f'  GPU {gpu} → {cfg["tag"]:<30}  '
               f'β={cfg["beta"]} η={cfg["eta"]} τ={cfg["tau"]} σ={cfg["sigma_start"]} '
               f'T={cfg["n_steps"]} auto_β={cfg["auto_beta"]} ig_var={cfg["ignore_variance"]}')
    print()

    # folder + run_config 미리 생성
    for gpu, cfg in CUSTOM_CONFIGS.items():
        cfg_core = {k: v for k, v in cfg.items() if k != 'tag'}
        cfg_core.setdefault('n_high', args.n_high)
        cfg_core.setdefault('seed', args.seed)
        out_dir = SWEEPS_DIR / cfg['tag']
        out_dir.mkdir(exist_ok=True)
        (out_dir / 'run_config.json').write_text(json.dumps({
            'overrides': {'_custom': True, '_gpu': gpu},
            'cfg': cfg_core, 'folder': cfg['tag'],
        }, indent=2, default=str))

    # ProcessPool tasks: (task_idx, ensemble_root, folder, class_c, gpu, cfg, sweeps_dir)
    custom_tasks = []
    for gpu, cfg in CUSTOM_CONFIGS.items():
        cfg_core = {k: v for k, v in cfg.items() if k != 'tag'}
        cfg_core.setdefault('n_high', args.n_high)
        cfg_core.setdefault('seed', args.seed)
        for c in args.classes:
            custom_tasks.append((len(custom_tasks), str(args.ensemble_root),
                                   cfg['tag'], c, gpu, cfg_core, str(SWEEPS_DIR)))

    t_total = time.time()
    total_tasks = len(custom_tasks)
    done_count = [0]

    def _fmt_time_c(s):
        s = int(s); m, s = divmod(s, 60); h, m = divmod(m, 60)
        return f'{h:d}h{m:02d}m{s:02d}s' if h else f'{m:d}m{s:02d}s' if m else f'{s:d}s'

    with ProcessPoolExecutor(max_workers=len(CUSTOM_CONFIGS)) as ex:
        futures = [ex.submit(_sgld_worker, t) for t in custom_tasks]
        for fut in as_completed(futures):
            r = fut.result()
            done_count[0] += 1
            elapsed = time.time() - t_total
            pct = done_count[0] / total_tasks * 100
            print(f'  [{pct:5.1f}%  {done_count[0]}/{total_tasks}]  '
                   f'{r["folder"]:<28}  c{r["class_c"]}  gpu={r["gpu"]}  '
                   f'pid={r["pid"]}  ({r["dt"]:.1f}s)  '
                   f'M={r["M_init"]:.3f}→{r["M_final"]:.3f}  '
                   f'| elapsed {_fmt_time_c(elapsed)}', flush=True)
    print(f'\n✓ DONE — total {_fmt_time_c(time.time()-t_total)}')


In [ ]:

def compute_M(folder: Path, class_c: int, cfg: dict):
  d = np.load(folder / f'diag_raw_c{class_c}.npz', allow_pickle=True)
  traj = d['mean']
  print(traj)
  
folder_path = '/home/work/JooKyung/TabEBM/experiments/ebms/20260415_214026_Distance_EBM/comparisons/sessions/20260417_165505_all_axes_baseline/sweeps/baseline'
run_cfg = json.loads((folder_path / 'run_config.json').read_text())['cfg']
compute_M('/home/work/JooKyung/TabEBM/experiments/ebms/20260415_214026_Distance_EBM/comparisons/sessions/20260417_165505_all_axes_baseline/sweeps/baseline',0, run_cfg)

## 2. 각 config 별 M-vs-N 곡선 계산 — trajectory subset 재사용 (~초 단위)

In [ ]:
def compute_M_vs_N_for_folder(folder: Path, class_c: int, cfg: dict):
    '''folder/diag_raw_c{c}.npz 의 trajectory 를 읽어 N_SWEEP 각 N 에서 M_mean 계산.'''
    d = np.load(folder / f'diag_raw_c{class_c}.npz', allow_pickle=True)
    traj = d['trajectory']   # (T+1, N_HIGH, d)
    class_dir = args.ensemble_root / f'c{class_c}'
    score_var_fn = ensemble_score_var_fn(class_dir, gpu=args.gpus[0])
    real_tensor = torch.from_numpy(np.load(class_dir / 'class_data.npz')['X_class']) \
                       .float().to(f'cuda:{args.gpus[0]}')
    beta_eff = (cfg['beta'] * compute_beta_scale(score_var_fn, real_tensor)
                  if cfg['auto_beta'] else cfg['beta'])

    out = {step_t: [] for step_t in EVAL_AT_STEPS}
    for N in N_SWEEP:
        for step_t in EVAL_AT_STEPS:
            step_clip = min(step_t, traj.shape[0] - 1)
            x_sub = torch.from_numpy(traj[step_clip, :N, :]).float() \
                          .to(f'cuda:{args.gpus[0]}')
            _, var = score_var_fn(x_sub)
            if cfg['ignore_variance']:
                M = torch.ones_like(var)
            else:
                M = 1.0 / (1.0 + beta_eff * var.clamp_min(1e-8))
            out[step_t].append(float(M.mean().item()))
    return out, beta_eff

m_vs_n = {}   # m_vs_n[folder_name][class][step] = list of M_mean
cfg_per_folder = {}

for folder_path in sorted(SWEEPS_DIR.iterdir()):
    if not folder_path.is_dir(): continue
    folder = folder_path.name
    run_cfg = json.loads((folder_path / 'run_config.json').read_text())['cfg']
    cfg_per_folder[folder] = run_cfg
    m_vs_n[folder] = {}
    for c in args.classes:
        out, beta_eff = compute_M_vs_N_for_folder(folder_path, c, run_cfg)
        m_vs_n[folder][c] = out

        # 저장: M vs N 데이터 + 작은 PNG
        rows = []
        for step_t in EVAL_AT_STEPS:
            for i, N in enumerate(N_SWEEP):
                rows.append({'N': N, 'step': step_t, 'M_mean': out[step_t][i]})
        pd.DataFrame(rows).to_csv(folder_path / f'M_vs_N_c{c}.csv', index=False)

        fig, ax = plt.subplots(figsize=(6, 4))
        for step_t in EVAL_AT_STEPS:
            ax.plot(N_SWEEP, out[step_t], marker='o', markersize=4,
                     label=f'step={step_t}')
        ax.set_xscale('log')
        ax.set_xlabel('N (chain count)')
        ax.set_ylabel('M_mean')
        ax.axhline(1.0, color='k', ls=':', lw=0.6, alpha=0.4)
        ax.set_ylim(-0.05, 1.1)
        ax.grid(alpha=0.25)
        ax.set_title(f'{folder}  ·  c{c}  ·  β_eff={beta_eff:.2e}')
        if len(EVAL_AT_STEPS) > 1: ax.legend(fontsize=8)
        plt.tight_layout()
        fig.savefig(folder_path / f'M_vs_N_c{c}.png', dpi=130, bbox_inches='tight')
        plt.close(fig)

print(f'{len(m_vs_n)} configs × {len(args.classes)} classes → M_vs_N_c{{c}}.{{csv, png}} 저장됨')

## 3. 각 sweep 의 EBM heatmap + 체인 plot

각 class 당 **heatmap 을 한 번만 계산** (viz_context 캐시) 후,
모든 sweep folder 의 final 체인 위치를 그 위에 찍어 저장.

출력: 각 `sweeps/<folder>/trajectory_c{c}.png`

In [ ]:
from viz_trajectory import precompute_class_viz_context

# --- 시각화 파라미터 ------------------------------------------------
VIZ_CHAINS_SHOW   = 20        # trajectory 선/화살표 그릴 체인 subset 크기
VIZ_BG            = 'std'     # 'std' | 'mean'  배경 EBM 종류
VIZ_BG_CMAP       = 'magma'
VIZ_TRAJ_CMAP     = 'turbo'
VIZ_REAL_COLOR    = 'cyan'
VIZ_NEG_COLOR     = 'red'
VIZ_FINAL_COLOR   = '0.15'
VIZ_SHOW_LINES    = False     # trajectory 선 (off 권장, arrow 만)
VIZ_SHOW_ARROWS   = True      # x_0 → x_T 방향 화살표
VIZ_HEATMAP_PAD   = 3.0       # heatmap 확장 pad (클수록 더 넓은 영역)
# -------------------------------------------------------------------

# class 당 heatmap + PCA + real/neg 한 번만 precompute (모든 sweep 이 재사용)
print('Precomputing per-class viz context (heatmap + PCA + negatives)...')
viz_ctx = {}
for c in args.classes:
    t0 = time.time()
    viz_ctx[c] = precompute_class_viz_context(
        args.ensemble_root / f'c{c}',
        pad=VIZ_HEATMAP_PAD, gpu=args.gpus[0],
    )
    print(f'  class {c}: heatmap shape {viz_ctx[c]["hm"]["std"].shape}  ({time.time()-t0:.1f}s)')

# 각 sweep folder 에 trajectory_c{c}.png 생성
print()
t_total = time.time()
n_done = 0
for folder_path in sorted(SWEEPS_DIR.iterdir()):
    if not folder_path.is_dir(): continue
    folder = folder_path.name
    cfg = json.loads((folder_path / 'run_config.json').read_text())['cfg']
    for c in args.classes:
        npz = folder_path / f'diag_raw_c{c}.npz'
        if not npz.exists():
            print(f'  [skip] {folder} c{c}: no npz'); continue
        d = np.load(npz, allow_pickle=True)
        traj = d['trajectory']                    # (T+1, N, d)
        ctx = viz_ctx[c]
        pca, Z_real, Z_neg, hm = ctx['pca'], ctx['Z_real'], ctx['Z_neg'], ctx['hm']
        xlim, ylim = ctx['xlim'], ctx['ylim']

        T_eff, B, d_dim = traj.shape
        Z_traj = pca.transform(traj.reshape(-1, d_dim)).reshape(T_eff, B, 2)
        Z_final = Z_traj[-1, :, :]

        rng = np.random.default_rng(0)
        show_idx = np.sort(rng.choice(B, min(VIZ_CHAINS_SHOW, B), replace=False))

        fig, ax = plt.subplots(figsize=(9, 7))
        cs = ax.contourf(hm['ZZ1'], hm['ZZ2'], hm[VIZ_BG], levels=24,
                          cmap=VIZ_BG_CMAP, alpha=0.85)
        cbar = fig.colorbar(cs, ax=ax, shrink=0.85); cbar.set_label(f'E_{VIZ_BG}', fontsize=9)
        ax.scatter(Z_real[:, 0], Z_real[:, 1], s=28, c=VIZ_REAL_COLOR,
                    edgecolors='black', linewidths=0.5, label='real',
                    alpha=0.85, zorder=4)
        if len(Z_neg):
            ax.scatter(Z_neg[:, 0], Z_neg[:, 1], s=80, c=VIZ_NEG_COLOR,
                        marker='+', linewidths=1.8,
                        label=f'neg (∪ K, {len(Z_neg)})', zorder=5)
        ax.scatter(Z_final[:, 0], Z_final[:, 1], s=14, c=VIZ_FINAL_COLOR,
                    edgecolors='white', linewidths=0.4,
                    label=f'final (N={B})', alpha=0.75, zorder=6)

        if VIZ_SHOW_LINES or VIZ_SHOW_ARROWS:
            cmap = plt.get_cmap(VIZ_TRAJ_CMAP)
            for j, i in enumerate(show_idx):
                col = cmap(j / max(1, len(show_idx) - 1))
                x = Z_traj[:, int(i), 0]; y = Z_traj[:, int(i), 1]
                if VIZ_SHOW_LINES:
                    ax.plot(x, y, color=col, linewidth=1.1, alpha=0.65, zorder=7)
                if VIZ_SHOW_ARROWS:
                    dx, dy = x[-1]-x[0], y[-1]-y[0]
                    if dx*dx + dy*dy > 1e-10:
                        ax.annotate('', xy=(x[-1], y[-1]), xytext=(x[0], y[0]),
                                      arrowprops=dict(arrowstyle='->', color=col,
                                                       lw=1.2, alpha=0.8,
                                                       shrinkA=0, shrinkB=0),
                                      zorder=8)

        ax.set_xlim(xlim); ax.set_ylim(ylim)
        ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
        ax.set_title(f'{folder}  ·  c{c}\n'
                      f"β={cfg['beta']}  η={cfg['eta']}  τ={cfg['tau']}  "
                      f"σ_start={cfg['sigma_start']}  T={cfg['n_steps']}  "
                      f"ig_var={cfg['ignore_variance']}",
                      fontsize=9)
        ax.legend(fontsize=8, loc='upper right', framealpha=0.85)
        ax.grid(alpha=0.2)
        plt.tight_layout()
        p = folder_path / f'trajectory_c{c}.png'
        fig.savefig(p, dpi=130, bbox_inches='tight')
        plt.close(fig)
        n_done += 1

print(f'{n_done} trajectory PNGs saved in {time.time()-t_total:.1f}s')

## 4. Axis 별 overlay plot — baseline + 한 축의 모든 값 겹쳐서

In [ ]:
# 각 axis 에 해당하는 folder 목록
axis_folders = {'baseline': ['baseline']}
for axis in SWEEPS:
    axis_folders[axis] = [f for f in m_vs_n.keys()
                            if f.startswith(f'{axis}__')]
    if 'baseline' not in axis_folders[axis]:
        axis_folders[axis] = ['baseline'] + axis_folders[axis]

n_axes = len([a for a in SWEEPS])
n_cols = 3
n_rows = (n_axes + n_cols - 1) // n_cols

for class_c in args.classes:
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 4*n_rows),
                               sharex=True, sharey=True, squeeze=False)
    for i, axis in enumerate(SWEEPS.keys()):
        ax = axes[i // n_cols, i % n_cols]
        folders = axis_folders[axis]
        n_folders = len(folders)
        cmap = plt.get_cmap('viridis', n_folders + 1)
        for j, folder in enumerate(folders):
            if folder not in m_vs_n: continue
            ys = m_vs_n[folder][class_c][EVAL_AT_STEPS[0]]
            label = folder if folder == 'baseline' else folder.split('__', 1)[1]
            color = 'k' if folder == 'baseline' else cmap(j)
            lw = 2.5 if folder == 'baseline' else 1.5
            ax.plot(N_SWEEP, ys, color=color, label=label,
                     linewidth=lw, marker='o', markersize=3)
        ax.set_xscale('log')
        ax.set_xlabel('N')
        ax.set_ylabel('M_mean')
        ax.axhline(1.0, color='k', ls=':', lw=0.5, alpha=0.4)
        ax.set_ylim(-0.05, 1.1)
        ax.grid(alpha=0.25)
        ax.set_title(f'sweep: {axis}')
        ax.legend(fontsize=7, loc='best')
    # hide empty subplots
    for k in range(len(SWEEPS), n_rows*n_cols):
        axes[k // n_cols, k % n_cols].axis('off')
    fig.suptitle(f'M vs N  |  class {class_c}  |  '
                  f'session: {SESSION_TS}_{args.session_tag}', fontsize=11, y=1.0)
    plt.tight_layout()
    p = SESSION_DIR / f'M_vs_N_overlay_c{class_c}.png'
    fig.savefig(p, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'  saved: {p}')

## 5. 세션 전체 summary

In [ ]:
rows = []
for folder, cfg in cfg_per_folder.items():
    axis = folder.split('__', 1)[0] if '__' in folder else folder
    value = folder.split('__', 1)[1] if '__' in folder else None
    for c in args.classes:
        d = np.load(SWEEPS_DIR / folder / f'diag_raw_c{c}.npz', allow_pickle=True)
        keys = list(d['diag_cols']); diag = d['diag']
        rows.append({
            'folder': folder, 'axis': axis, 'value': value, 'class': c,
            **{k: cfg[k] for k in ['beta','eta','tau','sigma_start','n_steps','auto_beta','ignore_variance']},
            'M_mean_init':  float(diag[0,  keys.index('M_mean')]),
            'M_mean_final': float(diag[-1, keys.index('M_mean')]),
            'M_min_final':  float(diag[-1, keys.index('M_min')]),
            'var_median_T': float(diag[-1, keys.index('var_median')]),
            'drift/noise_T':float(diag[-1, keys.index('drift_over_noise')]),
        })
df = pd.DataFrame(rows)
df.to_csv(SESSION_DIR / 'summary.csv', index=False)
print(f'saved: {SESSION_DIR / "summary.csv"}  ({len(df)} rows)')
df

## 6. 세션 디렉토리 나열

In [ ]:
print(f'=== SESSION: {SESSION_DIR} ===\n')
for f in sorted(SESSION_DIR.iterdir()):
    if f.is_file():
        print(f'  {f.name:<36} {f.stat().st_size:>10} bytes')
print()
print(f'=== SWEEPS: {SWEEPS_DIR} ===')
for sf in sorted(SWEEPS_DIR.iterdir()):
    if sf.is_dir():
        contents = sorted(p.name for p in sf.iterdir())
        print(f'  {sf.name}/')
        for fn in contents:
            size = (sf / fn).stat().st_size
            print(f'    {fn:<36} {size:>10} bytes')
print(f'\n=== 다른 세션들 (이전 run) ===')
sessions_root = args.ensemble_root / 'comparisons' / 'sessions'
for s in sorted(sessions_root.iterdir()):
    if s.is_dir() and s != SESSION_DIR:
        print(f'  {s.name}')